In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_kpi_schema, get_mart_schema, get_semantic_layer_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
mart_schema = get_mart_schema()
semantic_layer_schema = get_semantic_layer_schema()  # e.g., dev_platform

from superstore_logger import get_superstore_logger, log_event
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_product_insights_kpi= get_superstore_logger("superstore_product_insights_kpi")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_product_insights_kpi,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="kpi view"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{mart_schema}.mart_product_performance"
]
missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, mart_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_product_insights_kpi, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create product insights kpi view.", layer="kpi view")
else:
    # -------------------------------
    # Create kpi View (Semantic Layer for Analysts)
    # -------------------------------
    spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.{semantic_layer_schema}.product_insights_kpi AS
    WITH ranked_products AS (
        SELECT
            product_id,
            product_name,
            category,
            sub_category,
            total_sales,
            total_profit,
            total_quantity,
            profit_margin,
            sales_velocity,
            loss_making_flag,
            RANK() OVER (PARTITION BY category ORDER BY total_sales DESC) AS top_in_category_rank,
            RANK() OVER (PARTITION BY category ORDER BY total_sales ASC) AS bottom_in_category_rank
        FROM {catalog}.{mart_schema}.mart_product_performance
    ),
    category_totals AS (
        SELECT
            category,
            SUM(total_sales) AS category_sales,
            SUM(total_quantity) AS category_quantity,
            AVG(profit_margin) AS avg_category_margin
        FROM {catalog}.{mart_schema}.mart_product_performance
        GROUP BY category
    )
    SELECT
        rp.product_id,
        rp.product_name,
        rp.category,
        rp.sub_category,
        
        -- Sales & Profit Metrics
        rp.total_sales,
        rp.total_profit,
        rp.profit_margin,
        rp.sales_velocity,
        rp.total_quantity,
        
        -- Performance Flags
        rp.loss_making_flag,
        rp.top_in_category_rank,
        rp.bottom_in_category_rank,
        
        -- Contribution to category
        ROUND(rp.total_sales / ct.category_sales * 100, 2) AS sales_contribution_pct,
        ROUND(rp.total_quantity / ct.category_quantity * 100, 2) AS quantity_contribution_pct,
        ct.avg_category_margin AS category_avg_margin

    FROM ranked_products rp
    LEFT JOIN category_totals ct
    ON rp.category = ct.category
    """)
log_event(logger_product_insights_kpi, "INFO", "Product_insights_kpi view created successfully.", layer="kpi view")